# 06 - State the Brief-A answer

**Purpose:** run the final Brief-A check from a fresh kernel and print one clear number with its comparison, definition and caveat.

Before running this notebook, restart the Jupyter kernel and run the required notebooks from the beginning. Do not rely on variables or checks from an earlier session. This notebook reads the saved Hypo A metric summary, checks the expected values, writes the board-number workbook and records the commit used for the run.


## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "6",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-07-01"),
    "period_last": ("2025-01-01", "2025-07-01"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

In [ ]:
summary_path = ROOT / "data" / "hypo_A" / "brief_a_iitgn_metric_summary.csv"
checks.check("Hypo A metric summary exists", summary_path.exists(), str(summary_path))
summary = pd.read_csv(summary_path)
last = summary.loc[summary["period"] == "H1 2025"].iloc[0]
this = summary.loc[summary["period"] == "H1 2026"].iloc[0]
tests_last, tests_this = int(last["test_lines"]), int(this["test_lines"])
growth = (tests_this / tests_last - 1) * 100
HEADLINE = {"claim": "H1 2026 test volume increased 5.0% versus H1 2025, with more people and more test lines per visit despite slightly fewer visits per person.", "number": round(growth, 1), "numerator": tests_this, "denominator": tests_last,
            "unit": "validated booking test lines after the documented IITGN validation and identity rules",
            "period": "H1 2026 against H1 2025, six cities", "comparison": "H1 2025 baseline: 433,611 test lines", "caveat": "This is a descriptive booking-line measure, not proof of attended or performed tests; it depends on the person rule and complete H1 coverage."}
for k, v in HEADLINE.items(): print(f"{k:>12}: {v}")
checks.check("H1 2025 denominator is 433,611", tests_last == 433611, tests_last)
checks.check("H1 2026 numerator is 455,383", tests_this == 455383, tests_this)
checks.check("headline growth is 5.02 percent", abs(growth - 5.02109033211795) < 1e-9, growth)


## The board number, in Excel, with its sentence beside it

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
wb = Workbook(); ws = wb.active; ws.title = "Board number"
rows = [("Kalpa Health - Brief " + CONFIG["brief"] + " - Group " + CONFIG["group"], ""), ("", ""),
        ("The number", f"{HEADLINE['number']:+.1f}%"), ("What it counts", HEADLINE["unit"]), ("Numerator", HEADLINE["numerator"]), ("Denominator", HEADLINE["denominator"]),
        ("Period", HEADLINE["period"]), ("Compared with", HEADLINE["comparison"]), ("The sentence", HEADLINE["claim"]), ("The caveat", HEADLINE["caveat"]),
        ("Formula check", "=ROUND((B5/B6-1)*100,1)")]
for r in rows: ws.append(list(r))
ws["A1"].font = Font(bold=True, size=14); ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11): row[1].alignment = Alignment(wrap_text=True, vertical="top")
(ROOT / "slides").mkdir(exist_ok=True)
wb.save(ROOT / "slides" / "board_number.xlsx")
print("saved slides/board_number.xlsx; the formula in B11 must agree with B3 when opened")

## The repeatability check

After a real fresh-environment run, record the runner, whether the environment was fresh, runtime, headline and whether it matches the previous run in `docs/decisions-log.md`. Do not invent these details in the notebook.


In [ ]:
import subprocess
try:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    sha = "no git"
print(f"commit {sha}  headline {HEADLINE['number']:+.1f}%  ({HEADLINE['numerator']:,} / {HEADLINE['denominator']:,})")
checks.check("board workbook folder exists", (ROOT / "slides").exists())
checks.check("headline is a finite number", np.isfinite(HEADLINE["number"]))
checks.check("the claim sentence has been written", not HEADLINE["claim"].startswith("<"))
checks.check("the current Brief-A result is loaded", (HEADLINE["numerator"], HEADLINE["denominator"]) == (455383, 433611))
checks.summary()


## The last cell

The number on slide 3 is the one printed above. If it is not, one of the two is wrong, and the panel will ask which.